In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ All libraries loaded successfully")

✅ All libraries loaded successfully


In [3]:
RAW_PATH = '../data/raw/'

# Load main dataset
df_main = pd.read_csv(RAW_PATH + 'retail_store_inventory.csv')

print(f"Shape: {df_main.shape}")
print(f"\nColumns: {df_main.columns.tolist()}")
print(f"\nFirst 3 rows:")
df_main.head(3)

Shape: (73100, 15)

Columns: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']

First 3 rows:


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer


In [4]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

print(f"\n📊 Shape: {df_main.shape[0]:,} rows × {df_main.shape[1]} columns")

print(f"\n🔍 Missing Values:")
missing = df_main.isnull().sum()
missing_pct = (missing / len(df_main) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print(f"\n📋 Data Types:")
print(df_main.dtypes)

print(f"\n📈 Numeric Summary:")
print(df_main.describe().round(2))

DATA QUALITY REPORT

📊 Shape: 73,100 rows × 15 columns

🔍 Missing Values:
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []

📋 Data Types:
Date                      str
Store ID                  str
Product ID                str
Category                  str
Region                    str
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition         str
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality               str
dtype: object

📈 Numeric Summary:
       Inventory Level  Units Sold  Units Ordered  Demand Forecast    Price  \
count         73100.00    73100.00       73100.00         73100.00 73100.00   
mean            274.47      136.46         110.00           141.49    55.14   
std             129.95      108.92          52.28           109.25    26.02   
min              50.00        0.00          20.00

In [5]:
# Print exact column names from your file
print("Your columns:", df_main.columns.tolist())

# We'll rename columns to a standard format in Phase 2
# For now just confirm data is loaded correctly
print(f"\nSample data:")
print(df_main.head(3).to_string())

Your columns: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']

Sample data:
         Date Store ID Product ID   Category Region  Inventory Level  Units Sold  Units Ordered  Demand Forecast  Price  Discount Weather Condition  Holiday/Promotion  Competitor Pricing Seasonality
0  2022-01-01     S001      P0001  Groceries  North              231         127             55           135.47  33.50        20             Rainy                  0               29.69      Autumn
1  2022-01-01     S001      P0002       Toys  South              204         150             66           144.04  63.01        20             Sunny                  0               66.16      Autumn
2  2022-01-01     S001      P0003       Toys   West              102          65             51            74.02  27.99        10             Sunny  

In [6]:
DB_PATH = '../data/processed/freshmart.db'
conn = sqlite3.connect(DB_PATH)

# Save raw data to database
df_main.to_sql('raw_inventory', conn, if_exists='replace', index=False)

print(f"✅ Database created at: {DB_PATH}")
print(f"✅ Raw data saved: {len(df_main):,} records")
print(f"\nColumns saved: {df_main.columns.tolist()}")

conn.close()

✅ Database created at: ../data/processed/freshmart.db
✅ Raw data saved: 73,100 records

Columns saved: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']


In [7]:
import sqlite3
import os

DB_PATH = '../data/processed/freshmart.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

schema_sql = """
CREATE TABLE IF NOT EXISTS products (
    product_id      TEXT PRIMARY KEY,
    product_name    TEXT NOT NULL,
    category        TEXT NOT NULL,
    subcategory     TEXT,
    unit_cost       REAL,
    unit_price      REAL,
    supplier_id     TEXT
);

CREATE TABLE IF NOT EXISTS inventory (
    inventory_id        TEXT PRIMARY KEY,
    product_id          TEXT,
    warehouse_id        TEXT,
    stock_quantity      INTEGER,
    reorder_point       INTEGER,
    reorder_quantity    INTEGER,
    last_updated        DATE,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE IF NOT EXISTS sales_transactions (
    transaction_id      TEXT PRIMARY KEY,
    product_id          TEXT,
    warehouse_id        TEXT,
    sale_date           DATE,
    quantity_sold       INTEGER,
    sale_amount         REAL,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE IF NOT EXISTS suppliers (
    supplier_id         TEXT PRIMARY KEY,
    supplier_name       TEXT,
    lead_time_days      INTEGER,
    reliability_score   REAL
);

CREATE TABLE IF NOT EXISTS warehouses (
    warehouse_id        TEXT PRIMARY KEY,
    warehouse_name      TEXT,
    location_city       TEXT,
    capacity            INTEGER
);
"""

# Run each CREATE TABLE statement
cursor.executescript(schema_sql)
conn.commit()

# Verify all tables were created
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("✅ Schema created successfully!")
print(f"\nTables in database:")
for table in tables:
    print(f"   → {table[0]}")

conn.close()

✅ Schema created successfully!

Tables in database:
   → products
   → inventory
   → sales_transactions
   → suppliers
   → warehouses
   → clean_inventory
   → reorder_recommendations
   → raw_inventory
